In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('spam.csv',encoding='ISO-8859-1')
print(df.shape)

In [ ]:
#df = pd.read_csv('spam.csv',encoding='ISO-8859-1',chunksize=1000)

In [ ]:
# for chunk in df:
#     print(chunk.shape)

In [ ]:
#  *** Workflow***

# 1. Data cleaning
# 2. EDA
# 3. Text Preprocessing
# 4. Model Building
# 5. Evaluation
# 6. Improvement
# 7. Website
# 8. Deploy

## 1. Data cleaning

In [ ]:
df.info()

In [ ]:
# observation unnames2 , unnamed 3,unnamed4 has very less value.so remove these cols
df.drop(columns = ["Unnamed: 2","Unnamed: 3","Unnamed: 4"],inplace=True)

In [ ]:
df.head()

In [ ]:
# renaing the cols title

df.rename(columns = {"v1":"output" , "v2":"messages"},inplace=True)

In [ ]:
df.head()

In [ ]:
# there are 2 types of output spam and ham .. mark them as 0 1

from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df["output"] = encoder.fit_transform(df['output'])

In [ ]:
df.head()

In [ ]:
#check for missing values

df.isna().sum() # no missing values.

In [ ]:
#check duplicate values

df.duplicated().sum()

In [ ]:
# remove duplicates.
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

## 2. EDA : Exploratory Data analysis.

In [ ]:
df.head()

In [ ]:
#check how many spam and ham msgs are there
df["output"].value_counts() # observation : imbalanced data as ham msgs (4516) > spam(653)

In [ ]:
# pplot it in pie chart
import matplotlib.pyplot as plt
plt.pie(df["output"].value_counts(),labels = ["ham","spam"],autopct = "%0.2f")
plt.show() # 88 % are ham msgs and 13 % are spam

In [ ]:
import nltk


In [ ]:
df["num_of_char"] = df["messages"].apply(len)

In [ ]:
df.head()

In [ ]:
nltk.download('words')

In [ ]:
 df["no_of_words"] = df["messages"].apply(lambda x: len(nltk.word_tokenize(x)))

In [ ]:
df.head()

In [ ]:
# ham msgs
df[df["output"]== 0][["num_of_char","no_of_words"]].describe()

In [ ]:
# spam msgs
df[df["output"]== 1][["num_of_char","no_of_words"]].describe()

In [ ]:
# clearly seen, for ham msgs the num of char (70) is less than spam msgs num of char (137)
# so spam msgs are bigger than ham msgs.

In [ ]:
import seaborn as sns
plt.figure(figsize=(10,6))
sns.histplot(df[df["output"]==0]["num_of_char"]) # for ham
sns.histplot(df[df["output"]==1]["num_of_char"]) # for spam

In [ ]:
# check corelation

df.corr(numeric_only = True)

In [ ]:
sns.heatmap(df.corr(numeric_only = True),annot = True)
# output and num of char correlation is high (0.38). 
# when we build model we will use num of char 

## 3. Data Preprocessing
- lower case
- tokenization
- remove special characters
- remove stop words
- stemming / lemmatization

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
import string
punctuation_marks = string.punctuation
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet') 


In [ ]:
punctuation_marks


In [ ]:
text = []
punctuation_marks = string.punctuation
lemmatized_text = []
lemmatizer = WordNetLemmatizer()
def text_preprocess(msg):
    msg = msg.lower()
    msg = nltk.word_tokenize(msg)
    temp=[]
    
    for word in msg:
        if word not in stopwords.words('english') and word not in punctuation_marks:
            temp.append(word)
    text = temp[:]
    temp.clear()
    lemmatized_text = [lemmatizer.lemmatize(words) for words in text]

    return " ".join(lemmatized_text)

In [ ]:
df.to_csv('updated-spam-detection.csv')

In [ ]:
df["processed_text"] = df["messages"].apply(text_preprocess)

In [ ]:
df.head()

In [ ]:
df["messages"][0]

In [ ]:
df["processed_text"][0]

### finding most common words use in the spam msgs



In [ ]:
spam_msgs = df[df["output"]==1]['processed_text'].tolist() #convert spam msgs in list

In [ ]:
spam_corpus = []
for msg in spam_msgs: # iterating through sentece
    for word in msg.split(): # iterating words in each sentence
        spam_corpus.append(word)

In [ ]:
len(spam_corpus)

In [ ]:
from collections import Counter
Counter(spam_corpus).most_common(40)

### finding most common words use in the ham msgs

In [ ]:
ham_corpus=[]
ham_msgs = df[df["output"]==0]["processed_text"].tolist()
for sentence in ham_msgs:
    for word in sentence.split():
        ham_corpus.append(word)
        
        

In [ ]:
len(ham_corpus)

In [ ]:
from collections import Counter
Counter(ham_corpus).most_common(40)

# 4. Model building

- step1: text 2 vector ( using bag of word or TF-idf)
- step2: build model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features = 2000)


In [ ]:
X = tfidf_vectorizer.fit_transform(df["processed_text"]).toarray()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler() # uing minmaxscaler instead of standerd scaler. cz standard scaler given negative value
X = scaler.fit_transform(X)

In [ ]:
X.shape

In [ ]:
y = df["output"].values

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.15,random_state = 42)

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score

In [ ]:
gnb = GaussianNB()
mnb =  MultinomialNB()
bnb = BernoulliNB()

In [ ]:
# Gaussian NB
gnb.fit(X_train,y_train)
Y_pred1= gnb.predict(X_test)

print('accuracy score of GaussianNB: ',accuracy_score(y_test,Y_pred1))
print('confusion_matrix of GaussianNB: ')
print(confusion_matrix(y_test,Y_pred1))
print('precision_score of GaussianNB: ',precision_score(y_test,Y_pred1))


In [ ]:
# Multinomial NB

mnb.fit(X_train,y_train)
Y_pred2 = mnb.predict(X_test)

print("accuracy score of MultinomialNB: ",accuracy_score(y_test,Y_pred2))
print("precision score of MultinomialNB: ",precision_score(y_test,Y_pred2))
print('confusion_matrix of MultinomialNB: ')
print(confusion_matrix(y_test,Y_pred2))

In [ ]:
# BernoulliNB
bnb.fit(X_train,y_train)
Y_pred3 = bnb.predict(X_test)

print("accuracy score of BernoulliNB: ",accuracy_score(y_test,Y_pred3))
print("precision score of BernoulliNB: ",precision_score(y_test,Y_pred3))
print('confusion_matrix of BernoulliNB: ')
print(confusion_matrix(y_test,Y_pred3))


In [ ]:
# Observation: Multinomial NB gives best result in terms of precision score. which is more imp here

In [ ]:
!pip install scikit-learn xgboost

In [ ]:
# Comparing other ML models with Multinomial model
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier


In [ ]:

logistic_model = LogisticRegression()
svc_model = SVC()
multinomial_nb_model = MultinomialNB()
decision_tree_model = DecisionTreeClassifier()
knn_model = KNeighborsClassifier()
adaboost_model = AdaBoostClassifier()
bagging_model = BaggingClassifier()
extra_trees_model = ExtraTreesClassifier()
gradient_boost_model = GradientBoostingClassifier()
xgb_model = XGBClassifier()


In [ ]:
classifiers = {
    'logisticRegression' : logistic_model,
    'SVC': svc_model ,
    'MultinomialNB':multinomial_nb_model,
    'Decition Tree':decision_tree_model,
    'KNN':knn_model ,
    'AdaBoost':adaboost_model,
    'Bagging':bagging_model,
    'ExtraTrees' : extra_trees_model,
    'Gradient_boost':gradient_boost_model ,
    'XGB': xgb_model
}

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score

In [ ]:
def train_classifier(clf,X_train,y_train,X_test,y_test):
    clf.fit(X_train,y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test,y_pred)
    precision = precision_score(y_test,y_pred)
    return accuracy,precision

In [ ]:
accuracy,precision = train_classifier(xgb_model,X_train,y_train,X_test,y_test)

In [ ]:
print(accuracy)
print(precision)

In [ ]:
accuracy_scores = []
precision_scores = []

for name,clf in classifiers.items():
    curr_accuracy,curr_precision = train_classifier(clf,X_train,y_train,X_test,y_test)
    print("For ",name)
    print("accuracy: ",curr_accuracy)
    print("precision: ",curr_precision)
    accuracy_scores.append(curr_accuracy)
    precision_scores.append(curr_precision)
    
    

In [ ]:
import pandas as pd
performance_df = pd.DataFrame({"Algorithms": classifiers.keys(), "Accuracy": accuracy_scores, "Precision": precision_scores}).sort_values('Precision',ascending=False)


In [ ]:
performance_df 

In [ ]:
# Voting classifier : to make combination of best performing model so that model can be approved.
svc = SVC()
etc = ExtraTreesClassifier()# ExtraTrees
mnb = MultinomialNB()

from sklearn.ensemble import VotingClassifier

In [ ]:
voting_clf = VotingClassifier(estimators = [('svm',svc),('nb',mnb),('et',etc)],voting="hard")

In [ ]:
voting_clf.fit(X_train,y_train)

In [ ]:
from sklearn.metrics import accuracy_score,precision_score

In [ ]:

y_pred = voting_clf.predict(X_test)
print("Accuracy: ",accuracy_score(y_test,y_pred))
print("Precision : ",precision_score(y_test,y_pred))

In [ ]:
# voting classifies gives best result till now.

In [ ]:
#pipeline;
# whenever we get new emain we have check 3 step:
# 1. text pre process : lower case, tokenization, lemmatization
# 2 text 2 vector
# 3 apply algorithm

In [ ]:
import pickle
pickle.dump(tfidf_vectorizer,open('vectorizer.pkl','wb'))
pickle.dump(voting_clf,open('model.pkl','wb'))